# From your files to an experiment

This tutorial loads behaviour from files on disk: it writes a config for them, checks it, finds the sessions, loads
some of them, shows what happens when a file is broken, and saves the result as a store. It first writes its own
small set of files, into a temporary folder; with your own data, start at the config and point `data_dir` at your
folder.

The files are what a rig might write: one folder per animal and per day, each holding a `trials.csv` with the tone
in kHz, the response as 'L', 'R' or 'N' (no response), the laser, the reaction time in milliseconds, each trial's
start time in seconds, and the training stage.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

root = Path(tempfile.mkdtemp())
rng = np.random.default_rng(1)
for animal_id in ('R01', 'R02'):
    for day, stage in (('2025_03_10', 'training'), ('2025_03_11', 'task'), ('2025_03_12', 'task')):
        folder = root / 'data' / animal_id / day
        folder.mkdir(parents=True)
        n = 150
        tone = np.round(2 ** rng.uniform(1, 4, n), 2)
        p_right = 1 / (1 + np.exp(-(np.log2(tone) - 3) / 0.3))
        response = np.where(rng.random(n) < p_right, 'R', 'L')
        response[rng.random(n) < 0.03] = 'N'
        pd.DataFrame({
            'TrialNum': np.arange(1, n + 1),
            'Tone_kHz': tone,
            'Response': response,
            'Laser': (rng.random(n) < 0.3).astype(int),
            'RT_ms': np.round(rng.lognormal(np.log(450), 0.3, n)),
            'TrialStart_s': np.round(np.cumsum(rng.uniform(4, 7, n)), 3),
            'Stage': stage,
        }).to_csv(folder / 'trials.csv', index=False)

sorted(p.relative_to(root).as_posix() for p in root.rglob('*.csv'))

## The config

The config says where the files are and how to read them. `reader:` finds the sessions: here one folder per animal
and per day, with the date in the folder's name. `task:` puts the tone on the library's axis: the boundary, 8 kHz,
goes to 0 and the range, 2 to 16 kHz, to −1 and +1, on a log scale (see
[The stimulus axis](../concepts/stimulus-axis.md)). `columns:` maps each column to a field: the response through
`mapping`, so 'L' is category A (0), 'R' category B (1) and 'N' a no-response; the reaction time from
milliseconds; and `t_trial_start` as an event time on the `rig` clock. `session_metadata:` reads the stage from
each file's first row. The [configuration guide](../config_guide.md) describes every key.

In [ ]:
config_text = """
project:
  name: tone categorisation

reader:
  type: csv_folders
  data_dir: data
  behaviour_file: trials.csv
  date_regex: '(\\d{4})_(\\d{2})_(\\d{2})'

task:
  boundary: 8.0
  stimulus_range: [2.0, 16.0]
  stimulus_scale: log
  choice_mapping:
    type: identity
    no_response_value: null

clock:
  name: rig
  unit: s

columns:
  trial_number: {csv_name: TrialNum, dtype: int}
  stimulus: {csv_name: Tone_kHz, dtype: float}
  choice: {csv_name: Response, dtype: float, mapping: {L: 0, R: 1, N: .nan}}
  opto_on: {csv_name: Laser, dtype: bool}
  reaction_time: {csv_name: RT_ms, dtype: float, unit: ms}
  t_trial_start: {csv_name: TrialStart_s, dtype: float}

session_metadata:
  stage: {csv_name: Stage, dtype: str}
"""
(root / 'config.yaml').write_text(config_text)

from behav_utils import load_config

config = load_config(root / 'config.yaml')
config.name, config.task.boundary, config.task.stimulus_scale

`load_config` checks every key, and refuses one it does not know rather than ignoring it:

In [ ]:
typo_text = config_text.replace('behaviour_file', 'behavior_file')
(root / 'typo.yaml').write_text(typo_text)
load_config(root / 'typo.yaml')

## Finding the sessions

`discover_sessions` lists what the reader finds, with the ids that loading will give, without reading any trials.
Its table is the place to choose sessions:

In [ ]:
from behav_utils import discover_sessions

table = discover_sessions(config)
table[['session_id', 'animal_id', 'date', 'stage', 'ignored']]

## Loading

Only the chosen sessions are read:

In [ ]:
from behav_utils import load_experiment

chosen = table[table.stage == 'task']
experiment = load_experiment(config, sessions=chosen)
experiment.summary()

Each trial keeps the stimulus as read beside its value on the axis, and times are in seconds:

In [ ]:
session = experiment.get_animal('R01').sessions[0]
trials = session.trials
pd.DataFrame({'stimulus_raw': trials.stimulus_raw[:5], 'stimulus': trials.stimulus[:5].round(3),
              'choice': trials.choice[:5], 'reaction_time': trials.reaction_time[:5],
              't_trial_start': trials.events['t_trial_start'][:5]})

## A broken file

Loading is strict by default. Suppose one file has lost its response column:

In [ ]:
broken = root / 'data' / 'R02' / '2025_03_12' / 'trials.csv'
pd.read_csv(broken).drop(columns='Response').to_csv(broken, index=False)

A full load then stops at that session, and the error names it:

In [ ]:
load_experiment(config)

With `strict=False` the session is skipped, with a warning, and the rest load:

In [ ]:
experiment = load_experiment(config, strict=False)
experiment.summary()

## The store

A store keeps a loaded experiment as Parquet and JSON files, so it loads again as it was without the original files
or config. Its manifest lists the sessions without opening any trials, so part of a store can be loaded. It needs
pyarrow, the `store` extra.

In [ ]:
experiment.save(root / 'store')

from behav_utils import load_store, read_manifest

manifest = read_manifest(root / 'store')
manifest[['session_id', 'animal_id', 'date']]

Only R01's sessions:

In [ ]:
part = load_store(root / 'store', sessions=manifest[manifest.animal_id == 'R01'])
part.summary()

## Where to go next

- The [2-AFC tutorial](two_afc.ipynb) analyses an experiment once it is loaded.
- The [configuration guide](../config_guide.md) lists every key, including the other readers and per-session task
  blocks.
- [Identities and time](../concepts/identity-and-time.md) explains the session ids and the clock.